# Ablation Study Methods Walkthrough

This notebook explains each analysis technique used in the ablation study, with minimal code snippets you can run interactively.

**Context.** We pruned Aya Expanse 8B from 32 to 16 layers using IFR-guided layer removal for en-es translation. COMET dropped from 0.582 to 0.315. After LoRA FT with KD, COMET recovered to 0.836. We want to understand *why* pruning hurts and *what* FT does.

**Models referenced throughout:**
- `base` — `CohereForAI/aya-expanse-8b` (unpruned, no FT, 32 layers)
- `pruned_only` — `experiments/results/IP_16_enes/pruned_model/` (16 layers, no FT)
- `pruned_ft_kd` — `experiments/results/I2_16_enes/finetuned/merged/` (16 layers + LoRA + KD)
- `full_ft_kd` — `experiments/results/B4_enes/finetuned/merged/` (32 layers + LoRA + KD)

**Techniques covered:**
1. Cross-model CKA (representation similarity)
2. Pairwise within-model CKA + effective rank (redundancy)
3. Logit lens (intermediate predictions)
4. Attention pattern extraction
5. Weight diff analysis
6. Output error categorization
7. LoRA recovery curves
8. Surgical fixes (norm rescale, linear probe, localized FT)

In [ ]:
import json
from pathlib import Path
import numpy as np

RESULTS = Path('results')  # relative to this notebook's location

## 1. Cross-model CKA

**Goal:** Measure how similar the residual stream representations are between two models at corresponding layers.

**Idea.** CKA (Centered Kernel Alignment) compares activation patterns produced by two networks (or two layers of one network) on the *same* inputs. It's invariant to linear transformations of the representations, so it measures *what* each layer encodes, not *how*.

**Math (linear CKA):**
$$\text{CKA}(X, Y) = \frac{\|X^\top Y\|_F^2}{\|X^\top X\|_F \cdot \|Y^\top Y\|_F}$$
where $X, Y$ are centered activation matrices of shape `(n_examples, d_model)`.

**Implementation** (`ablation/scripts/cka.py`):

In [ ]:
def linear_cka(X, Y):
    X = X - X.mean(axis=0)
    Y = Y - Y.mean(axis=0)
    hsic_xy = np.linalg.norm(X.T @ Y, 'fro') ** 2
    hsic_xx = np.linalg.norm(X.T @ X, 'fro') ** 2
    hsic_yy = np.linalg.norm(Y.T @ Y, 'fro') ** 2
    return hsic_xy / np.sqrt(hsic_xx * hsic_yy)

# Load stored matched-depth CKA results
with open(RESULTS / 'hidden_state_divergence.json') as f:
    hs = json.load(f)

matched = hs['matched_pruned_only_vs_base']
for m in matched[-5:]:
    print(f"pruned layer {m['pruned_layer']:2d} vs full layer {m['full_layer']:2d}: CKA = {m['cka']:.4f}")

**Key finding.** Matched-depth CKA is >0.99 for most layers; only the final layer drops (to 0.87). The pruned model's internal representation *manifold* is nearly identical to the base model's at matched relative depths.

**How we collect activations from HuggingFace models.** Since TransformerLens doesn't support Cohere, we register forward hooks directly on each transformer block:

```python
def collect_residual_states(model, input_ids, attention_mask):
    activations = {}
    hooks = []
    for i, layer in enumerate(model.model.layers):
        def make_hook(idx):
            def hook(module, input, output):
                hidden = output[0] if isinstance(output, tuple) else output
                activations[idx] = hidden.detach()
            return hook
        hooks.append(layer.register_forward_hook(make_hook(i)))
    with torch.no_grad():
        model(input_ids=input_ids, attention_mask=attention_mask)
    for h in hooks: h.remove()
    # Take last non-padding token per example
    return activations
```

We capture the last non-padding token position because that's what the LM head ultimately uses to produce the next-token prediction.

## 2. Pairwise within-model CKA + effective rank

**Goal:** Measure how redundant adjacent layers are within a single model.

**Implementation.** Compute CKA between every pair of layers in the same model, producing an `(n_layers, n_layers)` matrix:

In [ ]:
pairwise_base = np.load(RESULTS / 'pairwise_cka_base.npy')
pairwise_full = np.load(RESULTS / 'pairwise_cka_full_ft_kd.npy')

# Adjacent-layer CKA (i, i+1) = how different is each layer from its neighbor?
adj_base = [pairwise_base[i, i+1] for i in range(31)]
print(f"Adjacent CKA in base model — min={min(adj_base):.4f}, max={max(adj_base):.4f}, mean={np.mean(adj_base):.4f}")
print(f"Middle-layer (5-25) adjacent CKA mean: {np.mean(adj_base[5:25]):.4f}")

**Effective rank.** For activations `A ∈ ℝ^{n × d}`, the effective rank is `exp(H(σ))` where `σ` are the normalized singular values of A and H is Shannon entropy. It tells us how many dimensions each layer actually uses.

```python
def effective_rank(activations):
    _, s, _ = np.linalg.svd(activations, full_matrices=False)
    s = s / s.sum()
    s = s[s > 1e-10]
    return np.exp(-np.sum(s * np.log(s)))
```

**Key finding.** Adjacent-layer CKA > 0.99 across all middle layers. Effective rank grows smoothly from ~10 (layer 1) to ~60 (layer 31). The middle layers of Aya 8B are genuinely redundant — they all represent roughly the same subspace with tiny incremental differences.

## 3. Logit lens

**Goal:** See what the model would predict at each intermediate layer, not just the final one.

**Idea.** The residual stream at layer `l` lives in the same space as the final residual. So we can apply the LM head to intermediate residuals and see what the model "thinks" at that layer.

**Critical dtype note.** Cohere uses bfloat16. Naive casting causes dtype mismatches. We cast both hidden state and LM head weights to float32 for the projection:

```python
logits = torch.nn.functional.linear(
    hidden.float(), lm_head.weight.float(),
    lm_head.bias.float() if lm_head.bias is not None else None,
)
```

In [ ]:
with open(RESULTS / 'logit_lens.json') as f:
    ll = json.load(f)

# Compare final-layer entropy (lower = more confident)
for name in ['base', 'pruned_only', 'pruned_ft_kd', 'full_ft_kd']:
    if name in ll:
        final_entropy = ll[name]['avg_entropy_per_layer'][-1]
        print(f"{name:<15} final-layer entropy: {final_entropy:.3f}")

**Key finding.** Pruned-only stays elevated at ~0.4-0.5 nats through late layers, while FT variants drop to ~0.2. The pruned model is uncertain about its output throughout generation.

## 4. Attention pattern extraction

**Goal:** Capture attention weight matrices from each layer.

**Gotcha.** Cohere uses SDPA (scaled dot-product attention) by default, which *doesn't* return attention weights through the output tuple. You must override the implementation:

```python
model.config._attn_implementation = 'eager'
for layer in model.model.layers:
    if hasattr(layer.self_attn, 'config'):
        layer.self_attn.config._attn_implementation = 'eager'
```

Then register a hook on each `layer.self_attn`:

```python
def hook(module, input, output):
    if isinstance(output, tuple) and len(output) >= 2 and output[1] is not None:
        attn_weights[idx] = output[1].detach().cpu().float().numpy()
```

**Metrics computed:**
- **Entropy**: `-sum(a * log(a))` over the attention distribution per query position, averaged across heads.
- **BOS fraction**: how much attention every query places on position 0 (attention-sink behavior).
- **Diagonal fraction**: self-attention (token attending to itself).

## 5. Weight diff analysis

**Goal:** Quantify how LoRA FT changed the weights, layer by layer.

**Important framing.** The I2_16 model's weights are the pruned base model's weights *with LoRA adapters merged in*. So the diff measures what LoRA learned.

**Implementation.** For each parameter tensor in common between pruned and FT models:

In [ ]:
with open(RESULTS / 'weight_diff_per_layer.json') as f:
    wd = json.load(f)

# Rank layers by how much they changed
by_change = sorted(wd, key=lambda x: -x['rel_change'])
print('Top 5 most-changed layers during LoRA FT:')
for d in by_change[:5]:
    print(f"  Layer {d['layer']:2d}: rel_change = {d['rel_change']:.4f}, attn_diff = {d['attn_diff']:.2f}, mlp_diff = {d['mlp_diff']:.2f}")

**Key finding.** Relative change per layer is only 2-3%. MLPs change more than attention. Early layers (0, 4, 5) change most.

**Why small per-layer changes?** LoRA is rank-constrained (r=16), so each adapter has at most ~130K degrees of freedom per projection — compared to ~16M for the full weight. The constraint forces minimal principled corrections.

## 6. Output error categorization

**Goal:** Classify the pruned model's translation errors (CPU-only, heuristic).

**Categories:**
- `wrong_language`: output contains more English markers than Spanish
- `verbose`: length ratio vs reference > 3
- `truncation`: length ratio < 0.3
- `repetition`: 3-gram uniqueness ratio < 0.5
- `source_copy`: hypothesis overlaps > 70% with source vocabulary
- `garbage`: output length < 5 chars
- `plausible`: none of the above

**Key finding.** Pruned-only: 100% wrong_language, 80% verbose. Both FT variants: 100% plausible.

## 7. LoRA recovery curves

**Goal:** Track COMET at intermediate checkpoints during LoRA FT.

**Implementation detail.** We save *adapter weights only* (~80MB) at each checkpoint, not merged models (8.5GB each). During evaluation we load the adapter, merge it into the base pruned model in GPU memory, evaluate, and discard — no extra disk I/O.

```python
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained(PRUNED_MODEL, torch_dtype=torch.bfloat16, device_map='cuda')
model = PeftModel.from_pretrained(base, adapter_dir)
model = model.merge_and_unload()
# ... translate + score ...
del model, base; torch.cuda.empty_cache()
```

We ran 5 variants: full_kd, no_kd, frac_0.25/0.5/0.75 (all authentic-only).

In [ ]:
import os
print('Recovery curve summary:')
print(f'{"Run":<12} {"Peak COMET":<12} {"Peak ckpt":<15} {"Final COMET":<12}')
for name in ['full_kd', 'no_kd', 'frac_0.25', 'frac_0.5', 'frac_0.75']:
    path = RESULTS / 'ft_recovery' / name / 'recovery_curve.json'
    if not path.exists():
        continue
    with open(path) as f:
        data = json.load(f)
    peak = max(data, key=lambda x: x['comet'])
    final = [d for d in data if d['checkpoint'] == 'final']
    final_comet = final[0]['comet'] if final else data[-1]['comet']
    print(f"{name:<12} {peak['comet']:<12.4f} {peak['checkpoint']:<15} {final_comet:<12.4f}")

## 8. Surgical fixes

**Goal:** Test whether a *localized* fix (at one layer) can recover quality, which would imply pruning only breaks the output readout.

### 8.1 RMSNorm rescale

**Hypothesis:** The pruned model's residual stream has different magnitudes than what the LM head expects. A scalar rescale fixes it.

```python
pruned_rms = np.sqrt((pruned_residuals ** 2).mean())
target_rms = np.sqrt((target_residuals ** 2).mean())
scale = target_rms / pruned_rms
model.model.norm.weight.data *= scale
```

**Result:** Scale = 1.057 (5.7% off). COMET = 0.292 (worse than baseline 0.315). Magnitude was never the problem.

### 8.2 Linear probe (closed-form)

**Hypothesis:** A learned linear map `T: h_pruned → h_target` at the pre-norm layer fixes the mismatch. Fit by ridge regression:

```python
def fit_linear_probe(X, Y, ridge=1e-3):
    d = X.shape[1]
    return np.linalg.solve(X.T @ X + ridge * np.eye(d), X.T @ Y)
```

Install as a pre-hook on `model.model.norm`.

**Result:** Reconstruction error = 0.0002 (nearly perfect fit), but COMET = 0.167 (terrible). The probe has 4096² params fit on only 500 samples — massively underdetermined. It memorizes training prompts but generalizes poorly.

### 8.3 LM head + norm FT

Freeze everything except `lm_head` and `model.model.norm`. Train with standard SFT loss on 5K examples, 1 epoch.

**Result:** COMET = 0.418. Only +0.10 over baseline despite training 1.05B params (23% of model).

### 8.4 Last-3 MLPs FT

Freeze everything except MLPs in layers 13, 14, 15 (the last 3 of 16).

**Result:** COMET = 0.430. Slightly better than 8.3, still far from the LoRA ceiling of 0.847.

### Why all 4 failed

They all concentrate the fix at one point in the network. But the problem is **compounded directional errors at every layer** during autoregressive generation — you can't undo that at the output. LoRA works because it places a small adapter at every layer, preventing the compounding.

## Summary: the unified story with the re-wiring analogy

Think of the transformer as a building:
- **Layers are rooms** (each has its learned contents = representations)
- **The residual stream is the hallway** running through all rooms
- **Projections (q, k, v, o, gate, up, down) are the doors** — each decides what the room reads from the hallway and what it writes back

**Pruning removes rooms.** The surviving rooms still have their contents (CKA > 0.99). The hallway still runs through them. But the doors were built for a 32-room building — their read/write patterns are calibrated for the specific hallway flow that only happens when all 32 rooms contribute. In a 16-room building, each door picks up subtly wrong signal.

**The wrongness compounds autoregressively.** Door misalignment at room 0 → residual error → room 1's door reads wrong signal → bigger error → ... → complete trajectory divergence by the output.

**LoRA re-aligns the doors.** Small rank-16 adjustments at every door. Not re-learning the room contents (representations) — just re-fitting the reading/writing interfaces to work in the shortened building.

**This is why LoRA recovery is fast.** 6% of training gets 89% of the way there. If we were re-learning representations, it would take much longer.

**Why each surgical attempt failed:**
- Localized (v1): "Adjust one door perfectly" — can't undo upstream misalignment
- Per-layer norm (v2): "Change hallway brightness" — doesn't fix doors
- Procrustes (v2): "Rotate each door by a fixed angle" — rotations can't stretch/compress; errors compound
- Closed-form rank-16 (v2): "Pre-fit doors for standard visitors" — fit for prompt positions, fails at generation positions
- Bias-only (v2): "Nudge each room's output slightly" — can shift outputs but can't change what each door reads
- **LoRA:** "Re-fit every door's read/write mechanism with gradient signal from real trajectories" — works.

See `ANALYSIS_REPORT.md` for full writeup with figures.

In [ ]:
# Compare all surgical attempts
with open(RESULTS / 'surgical_fix/surgical_all.json') as f:
    v1 = json.load(f)
with open(RESULTS / 'surgical_fix_v2/surgical_v2_all.json') as f:
    v2 = json.load(f)

print(f"{'Approach':<24} {'Series':<6} {'COMET':<8} {'vs baseline':<12}")
print('-' * 55)
baseline = 0.315
for r in v1:
    delta = r['comet'] - baseline
    print(f"{r['approach']:<24} {'v1':<6} {r['comet']:<8.4f} {delta:+.3f}")
for r in v2:
    delta = r['comet'] - baseline
    print(f"{r['approach']:<24} {'v2':<6} {r['comet']:<8.4f} {delta:+.3f}")
print(f"{'full_lora_no_kd':<24} {'ref':<6} {0.847:<8.4f} {0.847-baseline:+.3f}")

## Summary: the unified story

1. **Aya 8B has enormous redundancy** — adjacent middle layers are >99% CKA-similar.
2. **IFR correctly identifies removable layers** based on residual-stream contribution.
3. **Pruning leaves gross structure intact** but introduces **small directional errors at every layer**.
4. **Those errors compound during generation**, producing wrong-language output, verbose rambling, repetition.
5. **LoRA works because it places rank-16 micro-corrections at every layer's every projection** — preventing compounding at the source.
6. **Localized fixes fail** because you can't undo compounded errors post-hoc at a single interface.
7. **KD is unnecessary here** — authentic-only (0.847) > with-KD (0.813). Data efficiency is high.
8. **The remaining 0.046 gap to ceiling** (0.847 vs 0.893) is real lost capacity, but small given 50% layer removal.

See `ANALYSIS_REPORT.md` for the full writeup with figures.